## Import Libraries

In [1]:
# Standard Libraries
import os
from urllib.parse import urlparse

# Data Science Libraries
import numpy as np
import pandas as pd

# Scikit-learn (Machine Learning)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, RepeatedKFold
from sklearn.ensemble import BaggingRegressor
from sklearn.metrics import root_mean_squared_error, mean_squared_error

# =============================
# Global Variables
# =============================
random_state = 42

# Format floats to dollars with commas
def format_to_dollar(amount):
    return "${:,.2f}".format(amount)

## Downloading File

In [2]:
# If file is not already downloaded
url = "https://www.cs.bu.edu/fac/snyder/cs505/Data/zillow_dataset.csv"

filename = os.path.basename(urlparse(url).path)

if not os.path.exists(filename):
    try:
        print("Downloading the file...")
        response = requests.get(url)
        response.raise_for_status()  # Raise an error for bad status codes
        with open(filename, "wb") as f:
            f.write(response.content)
        print("File downloaded successfully.")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading the file: {e}")
else:
    print("File already exists. Skipping download.")

df = pd.read_csv(filename)

File already exists. Skipping download.


## Condensed Data Cleaning

In [3]:
# Make a copy of the original dataframe to preserve raw data
df_cleaned = df.copy()

# Drop features that are clearly not useful for regression modeling
features_to_drop = [
    'parcelid',                 # Unique identifier – not predictive
    'assessmentyear',           # Constant (2016 for all rows)
    'censustractandblock',      # Too granular, high cardinality
    'rawcensustractandblock',   # Same as above
    'longitude', 'latitude'     # High-cardinality raw geo features – better used with location clustering/engineering
]

# Drop the columns
df_cleaned.drop(columns=features_to_drop, inplace=True)

# Display remaining columns for confirmation
print("Remaining features after dropping unsuitable columns:")
print(df_cleaned.columns.tolist())

Remaining features after dropping unsuitable columns:
['airconditioningtypeid', 'architecturalstyletypeid', 'basementsqft', 'bathroomcnt', 'bedroomcnt', 'buildingclasstypeid', 'buildingqualitytypeid', 'calculatedbathnbr', 'decktypeid', 'finishedfloor1squarefeet', 'calculatedfinishedsquarefeet', 'finishedsquarefeet12', 'finishedsquarefeet13', 'finishedsquarefeet15', 'finishedsquarefeet50', 'finishedsquarefeet6', 'fips', 'fireplacecnt', 'fullbathcnt', 'garagecarcnt', 'garagetotalsqft', 'hashottuborspa', 'heatingorsystemtypeid', 'lotsizesquarefeet', 'poolcnt', 'poolsizesum', 'pooltypeid10', 'pooltypeid2', 'pooltypeid7', 'propertycountylandusecode', 'propertylandusetypeid', 'propertyzoningdesc', 'regionidcity', 'regionidcounty', 'regionidneighborhood', 'regionidzip', 'roomcnt', 'storytypeid', 'threequarterbathnbr', 'typeconstructiontypeid', 'unitcnt', 'yardbuildingsqft17', 'yardbuildingsqft26', 'yearbuilt', 'numberofstories', 'fireplaceflag', 'taxdelinquencyflag', 'taxdelinquencyyear', 'ta

In [4]:
# Set a missing value threshold (e.g., drop columns with >60% missing values)
missing_threshold = 0.60

# Identify features to drop
high_null_cols = df_cleaned.columns[df_cleaned.isnull().mean() > missing_threshold].tolist()

print(f"Columns to drop due to >{int(missing_threshold * 100)}% missing values:")
print(high_null_cols)

# Drop them
df_cleaned.drop(columns=high_null_cols, inplace=True)

# Show remaining shape
print(f"\nRemaining shape after dropping high-null columns: {df_cleaned.shape}")

Columns to drop due to >60% missing values:
['airconditioningtypeid', 'architecturalstyletypeid', 'basementsqft', 'buildingclasstypeid', 'decktypeid', 'finishedfloor1squarefeet', 'finishedsquarefeet13', 'finishedsquarefeet15', 'finishedsquarefeet50', 'finishedsquarefeet6', 'fireplacecnt', 'garagecarcnt', 'garagetotalsqft', 'hashottuborspa', 'poolcnt', 'poolsizesum', 'pooltypeid10', 'pooltypeid2', 'pooltypeid7', 'regionidneighborhood', 'storytypeid', 'threequarterbathnbr', 'typeconstructiontypeid', 'yardbuildingsqft17', 'yardbuildingsqft26', 'numberofstories', 'fireplaceflag', 'taxdelinquencyflag', 'taxdelinquencyyear']

Remaining shape after dropping high-null columns: (77613, 20)


In [5]:
# 1. Drop rows with null target value
df_cleaned = df_cleaned[df_cleaned['taxvaluedollarcnt'].notnull()]

# 2. Drop rows with too many missing features (more than 50% missing)
row_missing_threshold = 0.5
df_cleaned = df_cleaned[df_cleaned.isnull().mean(axis=1) <= row_missing_threshold]

# 3. Drop outliers in the target variable
# Let's look at 99.5 percentile and drop anything beyond that (e.g., extreme high-end properties)
high_value_threshold = df_cleaned['taxvaluedollarcnt'].quantile(0.995)
df_cleaned = df_cleaned[df_cleaned['taxvaluedollarcnt'] <= high_value_threshold]

# Final shape after filtering
print(f"Remaining samples after dropping problematic rows: {df_cleaned.shape}")

Remaining samples after dropping problematic rows: (77185, 20)


In [6]:
# Impute missing values

# Separate numerical and categorical columns
cat_cols = ["buildingqualitytypeid", "fips", "heatingorsystemtypeid", "propertycountylandusecode", "propertylandusetypeid", "propertyzoningdesc", "regionidcity", "regionidcounty", "regionidzip"]
num_cols = list(set(df_cleaned.columns) - set(cat_cols))

# Median imputation for numerical columns (robust to outliers)
num_imputer = SimpleImputer(strategy='median')
df_cleaned[num_cols] = num_imputer.fit_transform(df_cleaned[num_cols])

# Mode imputation for categorical columns
cat_imputer = SimpleImputer(strategy='most_frequent')
df_cleaned[cat_cols] = cat_imputer.fit_transform(df_cleaned[cat_cols])

# === Final check ===
print("Remaining missing values in dataset:", df_cleaned.isnull().sum().sum())

Remaining missing values in dataset: 0


In [7]:
# Identify categorical columns (object dtype)
categorical_cols = df_cleaned.select_dtypes(include=['object']).columns.tolist()

# Encode if there are any categorical columns
if categorical_cols:
    encoder = LabelEncoder()
    for col in categorical_cols:
        df_cleaned[col] = encoder.fit_transform(df_cleaned[col])
    print("Categorical columns encoded:", categorical_cols)
else:
    print("No categorical columns found to encode.")

Categorical columns encoded: ['buildingqualitytypeid', 'fips', 'heatingorsystemtypeid', 'propertycountylandusecode', 'propertylandusetypeid', 'propertyzoningdesc', 'regionidcity', 'regionidcounty', 'regionidzip']


## Feature Engineering

In [8]:
df = df_cleaned.copy()

# Split into input features and target feature
X = df.drop(columns=["taxvaluedollarcnt"])
y = df["taxvaluedollarcnt"]

In [9]:
# Split into training set and testing set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, train_size=0.8, random_state=random_state)

In [10]:
# Add polynomial terms
X_train["sqft_squared"] = X_train["calculatedfinishedsquarefeet"] ** 2
X_train["sqft_bathroom_interaction"] = X_train["calculatedfinishedsquarefeet"] * X_train["bathroomcnt"]
X_test["sqft_squared"] = X_test["calculatedfinishedsquarefeet"] ** 2
X_test["sqft_bathroom_interaction"] = X_test["calculatedfinishedsquarefeet"] * X_test["bathroomcnt"]

# Add log terms
X_train["log_finishedsquarefeet12"] = np.log(X_train["finishedsquarefeet12"])
X_train["log_calculatedfinishedsquarefeet"] = np.log(X_train["calculatedfinishedsquarefeet"])
X_test["log_finishedsquarefeet12"] = np.log(X_test["finishedsquarefeet12"])
X_test["log_calculatedfinishedsquarefeet"] = np.log(X_test["calculatedfinishedsquarefeet"])

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

# Scale using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Final Model

In [11]:
# Best feature set after feature selection on bagging regressor model
br_X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns)[['regionidzip', 'bathroomcnt', 'propertylandusetypeid', 'fullbathcnt', 'regionidcounty', 'fips', 'calculatedbathnbr', 'unitcnt', 'heatingorsystemtypeid', 'regionidcity', 'buildingqualitytypeid', 'finishedsquarefeet12', 'yearbuilt', 'lotsizesquarefeet', 'propertyzoningdesc', 'roomcnt', 'propertycountylandusecode', 'sqft_squared']]
br_X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns)[['regionidzip', 'bathroomcnt', 'propertylandusetypeid', 'fullbathcnt', 'regionidcounty', 'fips', 'calculatedbathnbr', 'unitcnt', 'heatingorsystemtypeid', 'regionidcity', 'buildingqualitytypeid', 'finishedsquarefeet12', 'yearbuilt', 'lotsizesquarefeet', 'propertyzoningdesc', 'roomcnt', 'propertycountylandusecode', 'sqft_squared']]

In [12]:
# Repeated cross-validation with default 5 folds and 10 repeats
repeated_cv = RepeatedKFold(random_state=random_state)

# Our version of the run_model from homework 7
def run_model(model, X_train, y_train, X_test, y_test, n_jobs=-1):
    mse_scores = -cross_val_score(model, X_train, y_train, scoring = 'neg_mean_squared_error', cv = repeated_cv, n_jobs=n_jobs)
    mean_cv_rmse = np.mean(np.sqrt(mse_scores))
    rmse_scores = np.sqrt(mse_scores)
    std_cv_rmse  = np.std(rmse_scores)
    
    # Fit the model on the training set
    model.fit(X_train, y_train)
    
    # Compute training MSE and testing MSE
    train_preds = model.predict(X_train)
    train_rmse   = root_mean_squared_error(y_train, train_preds)
    test_preds  = model.predict(X_test)
    test_rmse    = root_mean_squared_error(y_test, test_preds)

    return mean_cv_rmse, std_cv_rmse, train_rmse, test_rmse

In [13]:
mean_cv_rmse, std_cv_rmse, train_rmse, test_rmse = run_model(BaggingRegressor(n_estimators=200, max_samples=0.4, max_features=13, bootstrap=False, random_state=42), br_X_train, y_train, br_X_test, y_test, n_jobs=2)

/home/jason8924/.local/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [14]:
print("Mean CV RMSE:", format_to_dollar(mean_cv_rmse))
print("Standard Deviation CV RMSE:", format_to_dollar(std_cv_rmse))
print("Training RMSE:", format_to_dollar(train_rmse))
print("Testing RMSE:", format_to_dollar(test_rmse))

Mean CV RMSE: $277,539.21
Standard Deviation CV RMSE: $3,379.59
Training RMSE: $167,509.23
Testing RMSE: $267,797.62


## Reconsiderations

### Milestone One

Initially, during preprocessing, we decided to drop columns with missing values in over 90% of the rows. However, after reviewing this step, we realized that we didn't have a complete view of the missing value percentages across all features. Our code sorted the features by missing value percentage and only displayed the top 20. This made us overlook how close the missing value percentages were until the significant drop from 60% to 36%. Based on that jump, we decided 60% was a more reasonable threshold. Also, even without that gap, 90% was probably too high anyway, since it would allow columns with only 10% of their rows filled to remain.

Now that we know tree-based ensemble models are generally better than the linear models on this dataset, we would reconsider the features added during feature engineering. Tree-based models are naturally good at handling non-linear relationships and log skewness, so the polynomial and log features that we added won't be too helpful towards our tree-based models. Instead, we could have focused more on the high-cardinality categorical features. An example is "propertyzoningdesc" a categorical feature describing the property's zoning with 1896 unique values. It is made up of a combination of several codes and it would have been more useful to our tree-based model if we split it up into several features.

When encoding our categorical features, we were unsure of how to deal with high-dimensionality categorical features. One-hot encoding didn't seem like a good idea with the high number of unique values and ordinal encoding didn't feel right when there was no inherent order to the values. However, one encoding technique that we didn't think of, but works well with tree-based models is frequency encoding. For a feature like "regionidzip", the frequency of each value could provide useful information, such as housing density, while also reducing dimensionality.

### Milestone Two



While reconsidering our model evaluation process, we thought that applying a log transformation to our target feature could be helpful, since it is highly right-skewed. This transformation would normalize the distribution, potentially improving the performance of our models and reducing the impact of outliers. After making predictions, we would then convert the predictions back to the original scale in order to evaluate the RMSE.